In [30]:
from os.path import join, exists
from os import mkdir
from torch.nn import functional as F
from torchvision import transforms
import torch as th
from PIL import Image
from matplotlib import pyplot as plt
from tqdm import tqdm

In [2]:
zip_path = "/home/samuel/Téléchargements/vesuvius-challenge-ink-detection.zip"

In [3]:
output_dir = "/home/samuel/PycharmProjects/VesuviusChallenge/res"

In [5]:
if not exists(output_dir):
    mkdir(output_dir)

In [ ]:
!unzip $zip_path -d $output_dir

In [ ]:
!ls $output_dir/train

In [20]:
to_tensor = transforms.ToTensor()

extracted_tensor_path = join(output_dir, "train_tensors")
if not exists(extracted_tensor_path):
    mkdir(extracted_tensor_path)

In [ ]:
DESIRED_SIZE = (256, 256)

idx = 0

for img_idx in range(1, 4):
    img_folder = join(output_dir, "train", str(img_idx))
    
    mask = join(img_folder, "inklabels.png")
    mask_t = (
        F.unfold(
            to_tensor(Image.open(mask))[None],
            DESIRED_SIZE, 1, 0, DESIRED_SIZE
        )
        .view(1, DESIRED_SIZE[0], DESIRED_SIZE[1], -1)
        .permute(3, 0, 1, 2)
        .gt(0)
        .to(th.uint8)
    )
    
    for i in tqdm(range(mask_t.size(0))):
        th.save(
            mask_t[i], join(extracted_tensor_path, f"mask_{idx + i}.pt")
        )
    
    idx += mask_t.size(0)

100%|██████████| 744/744 [18:41<00:00,  1.51s/it]  
/home/samuel/PycharmProjects/VesuviusChallenge/venv/lib/python3.11/site-packages/PIL/Image.py:3182: DecompressionBombWarning: Image size (140973980 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
 49%|████▉     | 1039/2109 [1:24:53<34:46,  1.95s/it]  

In [14]:
# TODO finish this cell (dtype, min / max scale)

idx = 0

for img_idx in range(1, 4):
    img_folder = join(output_dir, "train", str(img_idx), "surface_volume")
    
    slices_t = []
    
    for slice_idx in range(1, 65):
        slice_path = join(img_folder, f"{slice_idx:2}.tif")
        
        slices_t.append(
            F.unfold(
                to_tensor(Image.open(slice_path)),
                DESIRED_SIZE, 1, 0, DESIRED_SIZE
            )
            .view(1, DESIRED_SIZE[0], DESIRED_SIZE[1], -1)
            .permute(3, 0, 1, 2)
        )
    
    slices_t = th.stack(slices_t, dim=4)
    
    for i in tqdm(range(slices_t.size(0))):
        th.save(
            slices_t[i], join(extracted_tensor_path, f"img_{idx + i}.pt")
        )
    
    idx += slices_t.size(0)